# 8. GPU 与多 GPU

GPU 是深度学习的核心加速硬件。PyTorch 让 GPU 的使用非常简单。

### 核心原则

模型和数据必须在同一个设备上，否则会报错。CPU 上的模型无法处理 GPU 上的数据，反之亦然。

In [5]:
import torch
import torch.nn as nn

## 8.1 检查 GPU 信息

在写代码前，先了解你的 GPU 情况。

In [6]:
# GPU 基本信息
print(f"CUDA 可用: {torch.cuda.is_available()}")
print(f"GPU 数量: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU 名称: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

CUDA 可用: True
GPU 数量: 1
GPU 名称: NVIDIA GeForce RTX 3060 Laptop GPU
GPU 显存: 6.0 GB


## 8.2 设备指定

推荐用这种方式自动选择设备，代码在不同机器上都能跑：

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
```

In [7]:
# 自动选择设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前设备: {device}")

# 张量移到 GPU
x = torch.randn(3, 4)
x_gpu = x.to(device)
print(f"CPU 张量设备: {x.device}")
print(f"GPU 张量设备: {x_gpu.device}")

当前设备: cuda
CPU 张量设备: cpu
GPU 张量设备: cuda:0


In [8]:
# 模型移到 GPU
model = nn.Linear(10, 5).to(device)
print(f"模型参数设备: {next(model.parameters()).device}")

# 前向传播时，数据也要在 GPU 上
x = torch.randn(2, 10).to(device)
output = model(x)
print(f"输出设备: {output.device}")

模型参数设备: cuda:0
输出设备: cuda:0


## 8.3 GPU 和 CPU 之间的转换

- .to('cuda') 或 .to(device) — CPU 到 GPU
- .to('cpu') 或 .cpu() — GPU 到 CPU

注意：GPU 上的张量不能直接转 NumPy，需要先 .cpu()。

In [9]:
# GPU 和 CPU 之间转换
x_gpu = torch.randn(3).to('cuda')
x_cpu = x_gpu.cpu()

print(f"GPU → CPU: {x_cpu.device}")
print(f"可以转 NumPy: {x_cpu.numpy()}")

# GPU 张量转 NumPy 的快捷方式
arr = x_gpu.cpu().numpy()
print(f"GPU → CPU → NumPy: {arr}")

GPU → CPU: cpu
可以转 NumPy: [-0.7836583  2.733278  -1.2470043]
GPU → CPU → NumPy: [-0.7836583  2.733278  -1.2470043]


## 8.4 GPU 加速效果对比

用矩阵乘法来直观感受 GPU 的速度优势。

In [10]:
import time

# 大矩阵乘法
size = 1000

# CPU
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

start = time.time()
for _ in range(100):
    c_cpu = a_cpu @ b_cpu
cpu_time = time.time() - start

# GPU
a_gpu = a_cpu.to('cuda')
b_gpu = b_cpu.to('cuda')

# 先跑一次预热（GPU 第一次计算会有初始化开销）
_ = a_gpu @ b_gpu

start = time.time()
for _ in range(100):
    c_gpu = a_gpu @ b_gpu
torch.cuda.synchronize()  # 等 GPU 算完
gpu_time = time.time() - start

print(f"CPU: {cpu_time:.3f} 秒")
print(f"GPU: {gpu_time:.3f} 秒")
print(f"加速比: {cpu_time / gpu_time:.1f}x")

CPU: 0.719 秒
GPU: 0.367 秒
加速比: 2.0x


## 8.5 多 GPU 训练（了解即可）

当有多张 GPU 时，可以把模型或数据分布到多张卡上并行计算。

### DataParallel（简单但效率一般）
```python
model = nn.DataParallel(model)  # 一行代码搞定
```

### DistributedDataParallel（推荐，效率更高）
大型训练（比如 GPT、LLaMA）使用的方式，配置更复杂但性能更好。

目前只有一张 GPU 的话不用关心这些，了解就行。